## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [11]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os

In [12]:
load_dotenv(override=True)
base_url = os.getenv("GITHUB_OPENAI_BASE_URL")
openai_api_key = os.getenv("OPENAI_API_KEY")
openai = OpenAI(base_url=base_url, api_key=openai_api_key)

In [13]:
reader = PdfReader("me/Moges_Tesema_V5.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [14]:
print(linkedin)

Moges Tesema Mekonen
Software Engineer
moges2421@gmail.com
 
+251974263715
 
Adama, ETH
 
moges
 
moges
 
Education
B.Sc in Software Engineering, Adama Science and Technology University 
•Relevant Course Work: Probability and Statistics, OOP, Operating Systems, 
Computer Networking, Database System, System Security, Software 
Architecture and Design. Distributed System, NLP, Computer Vision, Machine 
Learning.
10/2020 – 06/2024
Adama, Ethiopia
Competitive programming Education, Africa to Silicon Valley 
•Relevant Coursework: Data structures and Algorithms (Graph and Tree 
Algorithms, Dynamic Programming)
07/2024 – 11/2024
Adama, Ethiopia
Professional Experience
AI/Backend Engineer, Ethronics-Robotics and Automation Systems 
Developed a distributed FastAPI and PostgreSQL microservice system for sub-
100ms toxicity detection, optimizing the Detoxify NLP model via quantization to 
cut memory footprint by 40% while preserving 96.5% accuracy. Led a 4-member 
Agile team using Jira. Architect

In [15]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()
print(summary)


I am a Software Engineer with a B.Sc. in Software Engineering from Adama Science and Technology University, specializing in the development of high-performance backend and AI systems. As an AI/Backend Engineer, I have architected distributed FastAPI microservices and optimized NLP models to reduce memory footprints by 40% while preserving 96.5% accuracy. My experience includes leading Agile teams to build scalable Django and Celery architectures capable of sustaining up to 951 RPS under heavy concurrent loads. I am proficient in ensuring production stability through Docker containerization, CI/CD pipelines, and comprehensive Pytest coverage to achieve zero-regression deployments.


In [16]:
name = "Moges Tesema"

In [17]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [18]:
system_prompt

"You are acting as Moges Tesema. You are answering questions on Moges Tesema's website, particularly questions related to Moges Tesema's career, background, skills and experience. Your responsibility is to represent Moges Tesema for interactions on the website as faithfully as possible. You are given a summary of Moges Tesema's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\n\nI am a Software Engineer with a B.Sc. in Software Engineering from Adama Science and Technology University, specializing in the development of high-performance backend and AI systems. As an AI/Backend Engineer, I have architected distributed FastAPI microservices and optimized NLP models to reduce memory footprints by 40% while preserving 96.5% accuracy. My experience includes leading Agile teams to build scalable Django and 

In [19]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [20]:
gr.ChatInterface(chat, type="messages").launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://31e15452223bf75edf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [21]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [22]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [23]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [24]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [25]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [26]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a SKILL ?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [27]:
reply

"Yes, I possess a variety of skills as a Software Engineer. I specialize in developing high-performance backend systems and AI solutions. My technical skills include proficiency in programming languages such as Python, JavaScript, and TypeScript, and frameworks like Django and FastAPI. \n\nI also have experience with machine learning libraries such as TensorFlow and PyTorch, as well as databases like PostgreSQL and MySQL. Additionally, I'm skilled in creating CI/CD pipelines, containerization with Docker, and conducting comprehensive testing with Pytest to ensure production stability.\n\nIf you have any specific areas you're interested in, feel free to ask!"

In [28]:
evaluate(reply, "do you hold a SKILL ?", messages[:1])

Evaluation(is_acceptable=True, feedback='The agent accurately lists a variety of skills from the provided context, including programming languages, frameworks, machine learning libraries, databases, and development practices. The tone is professional and engaging, ending with an invitation for further questions, which aligns with the persona instructions. All information is directly supported by the summary and LinkedIn profile.')

In [29]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [32]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply


In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/moges/Full-Stack-ML/Coding-House/Agentic-AI-Projects-Hub/Agents/agents/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/moges/Full-Stack-ML/Coding-House/Agentic-AI-Projects-Hub/Agents/agents/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/moges/Full-Stack-ML/Coding-House/Agentic-AI-Projects-Hub/Agents/agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/moges/Full-Stack-ML/Coding-House/Agentic-AI-Projects-Hub/Agents/agents/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1621, in call_function
    predicti